# DuckDB UI notebook SQL exporter

Export SQL cells from DuckDB UI notebooks stored in `ui.db`.

**NOTE:** Close DuckDB UI before running this notebook if DuckDB reports that `ui.db` is locked.

In [1]:
from pathlib import Path

# DuckDB UI notebook database.
#UI_DB_PATH = Path(r"C:\Users\bdill\.duckdb\extension_data\ui\ui.db")
UI_DB_PATH = Path.home() / ".duckdb" / "extension_data" / "ui" / "ui.db"

# Folder where .sql exports will be saved.
EXPORT_DIR = Path(r"D:\Codex\Python-DuckDB_notebook_exporter\sql")

# Notebook titles to export.
NOTEBOOKS_TO_EXPORT = []  # empty list will export ALL notebooks
NOTEBOOKS_TO_EXPORT = ["Olympics", "Billboard Hot 100"]

In [2]:
import json
import re
from datetime import datetime

import duckdb
import pandas as pd
from IPython.display import display

def safe_filename(value: str, max_length: int = 180) -> str:
    """Return a Windows-safe filename stem."""
    cleaned = re.sub(r'[<>:"/\\|?*\x00-\x1F]', "_", value).rstrip(" .")
    return (cleaned[:max_length] or "notebook")


def cell_to_sql(cell: dict, index: int) -> str | None:
    query = cell.get("query")
    if query is None or not str(query).strip():
        return None

    query = str(query).replace("\u00a0", " ").rstrip()
    suffix = "" if query.rstrip().endswith(";") else ";"
    return f"-- ================================================================================\n-- Cell {index}\n{query}{suffix}"


def notebook_json_to_sql(payload: str) -> tuple[str, int]:
    notebook = json.loads(payload)
    statements = []

    for index, cell in enumerate(notebook.get("cells", []), start=1):
        statement = cell_to_sql(cell, index)
        if statement:
            statements.append(statement)

    return "\n\n".join(statements), len(statements)

latest_notebooks_query = """
SELECT notebook_id, title, version, json, created
FROM notebook_versions
WHERE expires IS NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY title ORDER BY version DESC, created DESC) = 1
ORDER BY title;
"""

with duckdb.connect(str(UI_DB_PATH), read_only=True) as con:
    notebooks_df = con.sql(latest_notebooks_query).df()

if NOTEBOOKS_TO_EXPORT:
    notebooks_df = notebooks_df[notebooks_df["title"].isin(NOTEBOOKS_TO_EXPORT)].copy()

display(notebooks_df.drop(columns=["json"]).head(5))

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
exported_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

exports = []

for row in notebooks_df.itertuples(index=False):
    sql, cell_count = notebook_json_to_sql(row.json)
    created_at = pd.Timestamp(row.created).strftime("%Y-%m-%d %H:%M:%S")
    header = "\n".join(
        [
            f"-- DuckDB notebook: {row.title}",
            f"-- Created:  {created_at}",
            f"-- Exported: {exported_at}",
        ]
    )
    sql = f"{header}\n\n{sql}" if sql else header
    export_path = EXPORT_DIR / f"{safe_filename(row.title)}.sql"
    export_path.write_text(sql + ("\n" if sql else ""), encoding="utf-8")

    exports.append(
        {
            "title": row.title,
            "version": row.version,
            "cell_count": cell_count,
            "export_path": str(export_path),
            "bytes": export_path.stat().st_size,
        }
    )

exports_df = pd.DataFrame(exports)
display(exports_df.head(5))

,notebook_id,title,version,created
3,5ade5e03-a647-4913-a95e-6197f8dc0782,Billboard Hot 100,35,2026-06-07 03:23:51.531642
8,a41a8833-0e8a-4fa8-a27d-036bb141ebf2,Olympics,549,2026-06-17 13:58:49.311737


,title,version,cell_count,export_path,bytes
0,Billboard Hot 100,35,4,D:\Codex\Python-DuckDB_notebook_exporter\sql\B...,908
1,Olympics,549,19,D:\Codex\Python-DuckDB_notebook_exporter\sql\O...,7203
